In [7]:
from langchain.agents import AgentExecutor, create_tool_calling_agent, tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# BaseCallbackHandler

# 언어 모델 기반 애플리케이션을 개발하는 과정에서 디버깅은 중요한 역할

# 1. 언어 모델은 입력에 따라 다양한 응답을 생성하므로 어떤 단계에서 어떤 입력에 어떤 출력을 했는지 관찰해야 함
# 2. 여러 Runnable을 연결해서 체인 형태로 실행하는 상황에서 실행의 흐름을 관찰할 수 있음

# BaseCallbackHandler의 메서드는 이벤트 핸들러 역할 → 특정 이벤트 발생 시점에 원하는 동작을 추가

In [ ]:

# 1. BaseCallbackHandler를 상속하는 클래스를 만듬
# 2. BaseCallbackHandler의 메서드 오버라이드

# on_chat_model_start(serialized, messages, *, ...)
# on_chain_start(serialized, inputs, *, run_id)
# on_retriever_start(serialized, query, *, run_id)
# on_tool_start(serialized, input_str, *, run_id)
# on_llm_start(serialized, prompts, *, run_id)
#---------------------------------------------------------------
# on_agent_finish(finish, *, run_id[, ...])
# on_chain_end(outputs, *, run_id[, parent_run_id])
# on_retriever_end(documents, *, run_id[, ...])
# on_llm_end(response, *, run_id[, parent_run_id])
# on_tool_end(output, *, run_id[, parent_run_id])
#---------------------------------------------------------------
# on_llm_error(error, *, run_id[, parent_run_id])
# on_chain_error(error, *, run_id[, parent_run_id])
# on_retriever_error(error, *, run_id[, ...])
# on_tool_error(error, *, run_id[, parent_run_id])
#---------------------------------------------------------------
# on_agent_action(action, *, run_id[, ...])
# on_custom_event(name, data, *, run_id[, ...])
# on_llm_new_token(token, *[, chunk, ...])
#---------------------------------------------------------------
# on_retry(retry_state, *, run_id[, parent_run_id])
# on_text(text, *, run_id[, parent_run_id])
#---------------------------------------------------------------

In [ ]:
####################################################################

In [ ]:
# on_tool_start(self, serialized, input_str, **kwargs)
# on_tool_end(self, output, **kwargs)

# on_tool_start: AgentExecutor에게 AgentAction가 전달되고 실제 도구 실행을 시작하기 직전에 호출
# on_tool_end: 도구 실행이 끝나고, 그 실행 결과가 반환된 직후 실행

In [ ]:
# on_agent_action(action, *, run_id[, ...])과
# on_agent_finish(self, finish, **kwargs)

# on_agent_action: 언어 모델의 출력을 AgentAction 객체로 변환한 직후 발생
# on_agent_finish: 언어 모델의 출력을 AgentFinish 객체로 변환한 직후 발생

In [ ]:
# on_llm_start(self, serialized, prompts, **kwargs)
# on_llm_end(self, response, **kwargs): outputs 대신 response

# on_llm_start()는 프롬프트가 최종적으로 준비되어 Agent가 언어 모델에 전달되기 직전
# 이 프롬프트는 일반 프롬프트, ReAct 스타일, tool_calls 스타일 모두 포함

# on_llm_end()은 언어 모델이 토큰 생성을 모두 끝내면 호출
# on_llm_new_token()는 언어 모델이 토큰을 하나씩 생성할 때마다 호출

In [ ]:
# 1. on_chain_* 은 Chain 추상 클래스(langchain.chains.base.Chain)를 상속받은 객체에서만 발생
# 따라서 Chain 클래스만 on_chain_start/on_chain_end를 트리거

# 2. RunnableSequence을 포함한 Runnable은 Chain을 상속 받은 것이 아님
# RunnableSequence는 A|B|C, A, B, C는 Runnable
# Chain 클래스: LLMChain, ConversationChain, SequentialChain

# 3. BaseCallbackHandler의 대상은 Runnable
# 따라서 원칙적으로는 on_chain_start/end가 Runnable에서 발생하지 않는 게 정상
# 그럼에도 불구하고 PromptTemplate, ChatPromptTemplate, RunnableLambda, RunnableSequence의 경우 on_chain_start 트리거
# 그 이유는, 역호환성 + tracing 일관성
# 역호환성: LangChain 0.1 시절에는 모든 게 Chain 기반, 현재는 모든 게 Runnable 기반
# tracing 일관성: 이전처럼 on_chain_start으로 시작하게 하자

# 따라서
# tracing 일관성을 유지하기 위해 RunnableSequence의 첫 실행 단계에서 on_chain_start를 트리거
# on_chain_end가 트리거 된다면 LCEL 이전 단계 스타일 코드를 의미
# LCEL 스타일의 핵심 개념은 모든 구성 요소가 Runnable + 파이프라인 연결인데 Runnable은 Chain 클래스가 아님

In [ ]:
# Agent의 경우 내부 콜백이 호출됨

# on_llm_start(), on_llm_end(), on_agent_action(), on_tool_start(), on_tool_end()가 반복

# on_llm_start(): 완성된 프롬프트가 LLM에 전달되기 직전
# on_llm_end(): LLM이 응답 생성을 완료한 직후에 호출되는 콜백 (응답에는 tool_calls도 포함)
# on_agent_action(): LLM 응답을 AgentAction으로 변환한 직후
# on_tool_start(): 툴 실행 직전
# on_tool_end(): 툴 실행 직후

# 언어 모델이 content만 포함된 최종 답변을 반환하는 경우에는 아래와 같이 진행
# on_llm_start(), on_llm_end(), on_agent_finish()

In [ ]:
# on_llm_start(): 완성된 프롬프트가 LLM에 전달되기 직전

# on_chat_model_start(self, serialized, messages, **kwargs)
# 완성된 프롬프트가 LLM에 전달되기 직전 (모델은 아직 응답을 생성하지 않은 상태)

In [ ]:
# serialized; 직렬화

# A 세상  ----  *   ----    B 세상(b)
# *: A. B 들 다 이해

# 직렬화: a => *, b => *

In [ ]:
# serialized에는 이 "이제 막 실행되려고 하는 대상의 메타 데이터" 정보가 들어있음
# "뭔가를 하려고 하는 시점에 그 뭔가에 대한 정보를 이벤트 핸들러에게 전달하는 것"
# serialized: 이제 막 어떤 runnable이 실행되는가?에 대한 요약 --- serialized에 대응되는 입력 파라미터가 있음
# outputs: runnable 실행 결과
#---------------------------------------------------------------
# 파라미터에 serialized가 포함된 메서드

# on_chain_start(self, serialized, inputs, **kwargs)
# serialized와 inputs

# on_llm_start(self, serialized, prompts, **kwargs)
# serialized와 prompts

# on_chat_model_start(self, serialized, messages, **kwargs)
# serialized와 messages

# on_tool_start(self, serialized, input_str, **kwargs)
# serialized와 input_str

# on_retriever_start(self, serialized, query, **kwargs)
# serialized와 query
#---------------------------------------------------------------
# 파라미터에 outputs가 포함된 메서드
# on_chain_end(self, serialized, outputs, **kwargs)
# on_llm_end(self, response, **kwargs): outputs 대신 response
# on_tool_end(self, output, **kwargs): outputs이 아닌 output
# on_retriever_end(self, documents, **kwargs): outputs가 아닌 documents
# on_agent_finish(self, finish, **kwargs): finish 객체는 내부적으로 return_values 필드에 output을 포함



# BaseCallbackHandler의 대상은 Runnable이고
# BaseCallbackHandler 메서드는 대상 Runnable을 기준으로 나누면
# LLM, Tool, Retriever, 그 외 Runnable

# serialized 파라미터는 해당 Runnable의 요약 정보가 들어 있는 딕셔너리
# LLM인 경우: 모델에 관련된 요약 정보
# Tool의 경우: Tool에 관련된 요약 정보
# Retriever의 경우: Retriever에 관련된 요약 정보
# 그 외 Runnable의 경우: Runnable에 관련된 요약 정보
# ※ serialize는 Runnable 자체가 아니라 Runnable을 직렬화한 메타 정보라는 점

# 종류에 상관없이 포함되는 필드
{
    "id": ["langchain", "prompts", "chat", "ChatPromptTemplate"],
    # 해당 콜백을 발생시킨 Runnable 객체 ID
    # 자동으로 결정되며 모듈+클래스 이름(import할 때 사용하는 계층 구조)
    # 단일 값인 경우는 해당 Runnable을 생성 시 'id' 필드에 단일 문자열을 할당하는 경우 등이 있음
    "type": "constructor",
    # 어떻게 생성되었는지, 어떤 역할을 하는지
    "name": "SomeRunnable",
    # 콜백을 발생시킨 객체의 클래스 이름
    # 해당 Runnable을 생성할 때 'name' 필드에 값을 지정
    "inputs": {...},
    # invoke()로 전달한 입력 데이터
}


# serialized로 직렬화한 Runnable에 전달된 객체 정보

# on_chain_start(self, serialized, inputs, **kwargs)
# serialized와 inputs
# serialized로 직렬화한 Runnable에 전달된 객체는 inputs

# on_llm_start(self, serialized, prompts, **kwargs)
# serialized와 prompts
# serialized로 직렬화한 Runnable에 전달된 객체는 prompts

# on_chat_model_start(self, serialized, messages, **kwargs)
# serialized와 messages
# serialized로 직렬화한 Runnable에 전달된 객체는 messages

# on_tool_start(self, serialized, input_str, **kwargs)
# serialized와 input_str
# serialized로 직렬화한 Runnable에 전달된 객체는 input_str

# on_retriever_start(self, serialized, query, **kwargs)
# serialized와 query
# serialized로 직렬화한 Runnable에 전달된 객체는 query





# 결과는 직렬화하지 않음 (단순 타입)

# on_agent_finish(finish, *, run_id[, ...])
# finish: AgentExecutor 실행 결과

# on_chain_end(outputs, *, run_id[, parent_run_id])
# outputs: LLM, Tool, Retriever를 제외한 모든 Runnable의 실제 실행 결과

# on_retriever_end(documents, *, run_id[, ...])
# documents: Retriever가 반환한 문서 리스트

# on_llm_end(response, *, run_id[, parent_run_id])
# response: LLM 호출 결과

# on_tool_end(output, *, run_id[, parent_run_id])
# output: Tool 실행 결과


In [ ]:
########################################################################################################

In [1]:
from langchain.callbacks.base import BaseCallbackHandler

class MyCallbackHandler(BaseCallbackHandler):
    def on_agent_action(self, action, **kwargs):
        print(f"[on_agent_action] action: {action}")

    def on_agent_finish(self, finish, **kwargs):
        print(f"[on_agent_finish] finish: {finish}")

    def on_tool_start(self, serialized, input_str, **kwargs):
        print(f"[on_tool_start] tool: {serialized}, input: {input_str}")

    def on_tool_end(self, output, **kwargs):
        print(f"[on_tool_end] tool: {output}")

    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"[on_llm_start] llm: {serialized}, input: {prompts}")
    
    def on_llm_end(self, response, **kwargs):
        print(f"[on_llm_end] output: {response}")

    def on_chat_model_start(self, serialized, messages, **kwargs):
        print(f"[on_chat_model_start] chat model: {serialized}, input: {messages}")

    def on_chain_start(self, serialized, inputs, **kwargs):

        runnable_id = None
        runnable_type = None
        runnable_name = None

        if isinstance(serialized, dict):
            if "id" in serialized:
                if isinstance(serialized["id"], list):
                    runnable_id = ".".join(serialized["id"])
                else:
                    runnable_id = str(serialized["id"])

            runnable_type = serialized.get("type")
            runnable_name = serialized.get("name")
        if runnable_id is not None:
            print(f"[on_chain_start] id={runnable_id}, type={runnable_type}, name={runnable_name}, inputs={inputs}")

    def on_chain_end(self, outputs, **kwargs):
        serialized = kwargs.get('serialized', {})
        runnable_id = None
        runnable_type = None
        runnable_name = None

        if isinstance(serialized, dict):
            if "id" in serialized:
                if isinstance(serialized["id"], list):
                    runnable_id = ".".join(serialized["id"])
                else:
                    runnable_id = str(serialized["id"])

            runnable_type = serialized.get("type")
            runnable_name = serialized.get("name")

        if runnable_id is not None:
            print(f"[on_chain_end] id={runnable_id}, type={runnable_type}, name={runnable_name}, outputs={outputs}")

c:\Users\bayesian\miniconda3\envs\py310_64\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
# serialized의 id 필드가 리스트가 아닌 경우
{
  "id": "ChatOpenAI",
  "name": "ChatOpenAI",
  "type": "chat_model"
}
# serialized의 id 필드가 리스트인 경우
from langchain.prompts.chat import ChatPromptTemplate
{
  "id": [
    "langchain",
    "prompts",
    "chat",
    "ChatPromptTemplate"
  ],
  "name": "ChatPromptTemplate",
  "type": "prompt_template"
}

# ".".join(["langchain", "prompts", "chat", "ChatPromptTemplate"])의 결과는
# 'langchain.prompts.chat.ChatPromptTemplate'

In [2]:
# on_llm_start
# on_llm_end

from langchain_core.prompts import PromptTemplate
from langchain_community.llms import OpenAI

# 프롬프트
prompt = PromptTemplate.from_template("Say hello to {name}")

llm = OpenAI()

# LCEL 체인 (파서 없이)
# chain = prompt
chain = prompt | llm # RunnableSequence

chain.invoke({"name": "LangChain"}, config={"callbacks": [MyCallbackHandler()]})

# chain = prompt의 경우 on_llm_start는 실행되지 않음
# 최종 프롬프트가 준비되어 Agent가 언어 모델에 전달되기 직전에 실행되는 것이 on_llm_start
# prompt | llm가 아니라 prompt만으로는 on_llm_start는 트리거 되지 않음

# LCEL 파이프라인 chain = prompt | llm에서는 on_chain_end가 트리거 되지 않음 (RunnableSequence)

C:\Users\bayesian\AppData\Local\Temp\ipykernel_23204\1266629593.py:10: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  llm = OpenAI()


[on_chain_start] id=langchain.prompts.prompt.PromptTemplate, type=constructor, name=PromptTemplate, inputs={'name': 'LangChain'}
[on_llm_start] llm: {'lc': 1, 'type': 'constructor', 'id': ['langchain', 'llms', 'openai', 'OpenAI'], 'kwargs': {'model_name': 'gpt-3.5-turbo-instruct', 'temperature': 0.7, 'max_tokens': 256, 'top_p': 1.0, 'n': 1, 'best_of': 1, 'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']}, 'openai_proxy': '', 'batch_size': 20, 'max_retries': 2, 'disallowed_special': 'all'}, 'name': 'OpenAI'}, input: ['Say hello to LangChain']
[on_llm_end] output: generations=[[Generation(text='\n\n\nHello LangChain! Nice to meet you. I am an AI digital assistant designed to communicate with humans in various languages. How can I assist you?', generation_info={'finish_reason': 'stop', 'logprobs': None})]] llm_output={'token_usage': {'total_tokens': 37, 'completion_tokens': 32, 'prompt_tokens': 5}, 'model_name': 'gpt-3.5-turbo-instruct'} run=None type='LLMResult'


'\n\n\nHello LangChain! Nice to meet you. I am an AI digital assistant designed to communicate with humans in various languages. How can I assist you?'

In [3]:
# on_chat_model_start

from langchain.callbacks.base import BaseCallbackHandler
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage


chat = ChatOpenAI(
    streaming=False,  # 스트리밍 여부는 상관없음
    callbacks=[MyCallbackHandler()],
    temperature=0
)

# 메시지 실행
response = chat.invoke([HumanMessage(content="안녕!")])
print("응답:", response.content)

# LLM이 언어모델인가? 채팅 모델인가?를 따져서

# 채팅 모델이면
# on_chat_model_start: 완성된 프롬프트가 LLM에 전달되기 직전

# 일반 언어모델이면
# on_llm_start(): 완성된 프롬프트가 LLM에 전달되기 직전

# 종료는 동일하게
# on_llm_end: LLM이 응답 생성을 완료한 직후에 호출되는 콜백 (응답에는 tool_calls도 포함)

C:\Users\bayesian\AppData\Local\Temp\ipykernel_23204\3904036516.py:8: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat = ChatOpenAI(


[on_chat_model_start] chat model: {'lc': 1, 'type': 'constructor', 'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'], 'kwargs': {'model_name': 'gpt-3.5-turbo', 'temperature': 0.0, 'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']}, 'openai_proxy': '', 'max_retries': 2, 'n': 1}, 'name': 'ChatOpenAI'}, input: [[HumanMessage(content='안녕!', additional_kwargs={}, response_metadata={})]]
[on_llm_end] output: generations=[[ChatGeneration(text='안녕하세요! 무엇을 도와드릴까요?', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 12, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_rea

In [ ]:
# on_llm_new_token(token, *[, chunk, ...])


# 대한민국의 수도는 어디지?
# ........
# 대한민국의 수도는 서울입니다.

# 언어 모델 출력은 토큰 단위로 생성 (토큰이 무엇인지 어떻게 만드는지는 언어 모델의 tokenizer가 담당)
# on_llm_new_token: 언어 모델이 응답을 생성하는 과정에서 새로운 토큰이 만들어질 때마다 실행
# 언어 모델의 streaming=True이어야 함

# tool_calls는 토큰 스트리밍에 해당되지 않음 → tool_calls는 한번에 JSON-like 객체를 리턴하는 것

In [4]:
from langchain.callbacks.base import BaseCallbackHandler
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage

# BaseCallbackHandler
class StreamingCallbackHandler(BaseCallbackHandler):
    def on_llm_new_token(self, token: str, **kwargs) -> None:
        """새로운 토큰이 생성될 때마다 호출됩니다."""
        print(token, end="", flush=True)

chat = ChatOpenAI(
    model="gpt-3.5-turbo",
    streaming=True,
    # streaming=True이 없다면 언어 모델은 응답을 모두 만든 뒤 리턴
    # streaming=True이 없다면 on_llm_new_token에 해당하는 이벤트 자체가 발생하지 않음
    
    # streaming=True로 설정하면 언어 모델은 토큰 단위로 결과를 리턴하고
    # on_llm_new_token 콜백 함수가 호출되는 것
    callbacks=[StreamingCallbackHandler()],
    # 이 언어 모델에서 발생하는 이벤트에 해당하는 콜백은 객체 StreamingCallbackHandler()의 멤버 함수
    temperature=0
)

prompt = "미국 대통령 조 바이든에 대해 간략히 설명해줘."
print(f"프롬프트: {prompt}\n")
print("LLM 응답 스트리밍 시작...")
response = chat.invoke(prompt)
print("\n\n스트리밍 완료.")

프롬프트: 미국 대통령 조 바이든에 대해 간략히 설명해줘.

LLM 응답 스트리밍 시작...
조 바이든은 미국의 46대 대통령으로, 민주당 출신이며 2021년 1월 20일에 취임했다. 이전에는 미국 부통령과 선거 상원의원을 역임했으며, 경제, 기후변화, 인종 문제 등을 중점으로 정책을 추진하고 있다. 바이든은 미국 역사상 가장 나이 많은 대통령으로서 취임했으며, 미국 정치 경력이 길고 다양하다. 현재는 코로나19 대응, 경제 회복, 기후변화 대응 등 다양한 과제에 집중하고 있다.

스트리밍 완료.


In [5]:
from langchain.callbacks.base import BaseCallbackHandler
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage

# 토큰을 모으는 역할을 언어 모델이 아닌 콜백에서 처리
class StreamingCallbackHandler2(BaseCallbackHandler):
    def __init__(self):
        self.final_response = ""

    def on_llm_new_token(self, token: str, **kwargs) -> None:
        """새로운 토큰이 생성될 때마다 호출됩니다."""
        self.final_response += token

    def get_final_response(self):
        """누적된 최종 응답을 반환하는 메서드."""
        return self.final_response

streaming_handler = StreamingCallbackHandler2()

chat = ChatOpenAI(
    model="gpt-3.5-turbo",
    streaming=True,
    callbacks=[streaming_handler],
    temperature=0
)

prompt = "미국 대통령 조 바이든에 대해 간략히 설명해줘."
response = chat.invoke(prompt)
# invoke()는 동기식(synchronous) 호출이라서 모든 스트리밍이 완료가 되야 종료
# 따라서 invoke()가 끝난 시점에서 on_llm_new_token() 콜백도 모든 토큰에 대해 호출을 마친 상태

accumulated_response = streaming_handler.get_final_response()
print("---")
print(f"누적된 최종 응답: {accumulated_response}")

---
누적된 최종 응답: 조 바이든은 미국의 46대 대통령으로, 민주당 출신이며 2021년 1월 20일에 취임했다. 이전에는 미국 부통령과 선거인을 역임했으며, 36년간 미국 상원의원으로 활동했다. 바이든은 중간주의자로 알려져 있으며, 기후변화 대응, 경제 회복, 사회통합 등을 중요한 정책으로 내세우고 있다. 현재는 코로나19 대응과 경제 회복을 최우선 과제로 삼고 있다.


In [8]:
from langchain_openai import ChatOpenAI

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱하는 툴입니다."""
    return a * b

tools = [multiply]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 유용한 AI 어시스턴트입니다."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

# llm = ChatOpenAI(model="gpt-4o")
llm = ChatOpenAI(model="gpt-4o").with_config({"callbacks": [MyCallbackHandler()]})


agent = create_tool_calling_agent(llm, tools, prompt)


agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
).with_config({"callbacks": [MyCallbackHandler()]}) # 반드시 리스트 형태로 콜백 핸들러를 전달

result = agent_executor.invoke({"input": "100에 5를 곱해줘."})
print(f"\nFinal Result: {result['output']}")


# Agent가 언어 모델에게 프롬프트를 제출하는 단계에서 PromptTemplate에서
# on_chain_end만 생략, on_chain_start는 호출되는 것
# 이 경우 Agent는 모델에게 프롬프트를 두 번 제출 (첫 번째는 최초 요청, 두 번째는 도구 호출의 결과를 전달)



> Entering new AgentExecutor chain...
[on_chain_start] id=langchain.prompts.chat.ChatPromptTemplate, type=constructor, name=ChatPromptTemplate, inputs={'input': '100에 5를 곱해줘.', 'intermediate_steps': [], 'agent_scratchpad': []}
[on_chat_model_start] chat model: {'lc': 1, 'type': 'constructor', 'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'], 'kwargs': {'model_name': 'gpt-4o', 'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']}, 'output_version': 'v0'}, 'name': 'ChatOpenAI'}, input: [[SystemMessage(content='당신은 유용한 AI 어시스턴트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='100에 5를 곱해줘.', additional_kwargs={}, response_metadata={})]]
[on_llm_end] output: generations=[[ChatGenerationChunk(generation_info={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cbf1785567', 'service_tier': 'default'}, message=AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_l790Kv

In [ ]:
# 1. on_chain_* 은 Chain 추상 클래스(langchain.chains.base.Chain)를 상속받은 객체에서만 발생
# 따라서 Chain 클래스만 on_chain_start/on_chain_end를 트리거
# 이전에 그랬다는 이야기

# 2. RunnableSequence을 포함한 Runnable은 Chain을 상속 받은 것이 아님
# RunnableSequence는 A|B|C, A, B, C는 Runnable
# Chain 클래스: LLMChain, ConversationChain, SequentialChain
# 당연한 이야기

# 3. BaseCallbackHandler의 대상은 Runnable
# 따라서 원칙적으로는 on_chain_start/end가 Runnable에서 발생하지 않는 게 정상
# 그럼에도 불구하고 PromptTemplate, ChatPromptTemplate, RunnableLambda, RunnableSequence의 경우 on_chain_start 트리거
# 그 이유는, 역호환성 + tracing 일관성
# 역호환성: LangChain 0.1 시절에는 모든 게 Chain 기반, 현재는 모든 게 Runnable 기반
# tracing 일관성: 이전처럼 on_chain_start으로 시작하게 하자

# 따라서
# tracing 일관성을 유지하기 위해 RunnableSequence의 첫 실행 단계에서 on_chain_start를 트리거
# on_chain_end가 트리거 된다면 LCEL 이전 단계 스타일 코드를 의미
# LCEL 스타일의 핵심 개념은 모든 구성 요소가 Runnable + 파이프라인 연결인데 Runnable은 Chain 클래스가 아님

In [ ]:
# LangChain의 초기 구조에서는 Chain 객체만이 추적 및 모니터링의 주요 대상
# on_chain_start/on_chain_end 이벤트도 Chain 클래스에 한정

# 최근 구조에서는 모든 실행 단위(Runnable)를 추적 대상으로 확장
# 이벤트 이름은 on_chain_start/on_chain_end이지만, 의미는 "실행 단위의 시작과 끝"으로 확장


# on_chain_start/on_chain_end 관련 원칙은 이전에는 대상이 chain이었는데 지금은 대상이 Runnable

# RunnableSequence = Runnable1 | Runnable2인 경우
# on_chain_start, RunnableSequence, on_chain_end
# + 
# on_chain_start, Runnable1, on_chain_end, on_chain_start, Runnable2, on_chain_end
# =
# on_chain_start, on_chain_start, Runnable1, on_chain_end, on_chain_start, Runnable2, on_chain_end, on_chain_end

# 실제로는
# 생략을 안하는 Runnable이 있고 하는 Runnable이 정해져 있음

# 생략 안하는 Runnable들:
# OutputParser: on_chain_start/end 둘 다 호출
# RunnableLambda: on_chain_start/end 호출

# 생략하는 Runnable들:
# LLM: on_chain_start/end 생략 → on_llm_start/end로 대체
# ChatModel: on_chain_start/end 생략 → on_chat_model_start/end로 대체
# Tool: on_chain_start/end 생략 → on_tool_start/end로 대체
# RunnableSequence: 전체를 감싸는 on_chain_start/end 생략

# 부분 생략하는 Runnable들:
# PromptTemplate: on_chain_end만 생략, on_chain_start는 호출
# AgentExecutor: on_chain_end만 생략, on_chain_start는 호출

# 왜 생략을 하는가?
# 대체로 시작을 추적하는 것은 중요하지만 종료 추적은 결과물만 얻으면 되므로 덜 중요
# 너무 많은 콜백을 호출하면 성능의 문제의 원인이 될 수 있음